# Phishing-Email SLM Fine-Tuning Starter (QLoRA) — Colab-Safe Version
**Project:** Small but Sufficient? — SLM vs. LLM phishing-email detection

This notebook covers **Weeks 5–6** of your project plan: naive LoRA fine-tuning and CoT-distilled LoRA fine-tuning of a small model on phishing-email classification, with checkpoints saved to **Google Drive** so a Colab disconnect never loses your trained adapter.

**How to use, step by step:**
1. Open this file in Google Colab (Colab → File → Upload notebook, or open from GitHub if you pushed the repo).
2. Runtime → Change runtime type → **T4 GPU** (free tier is enough).
3. Run Cell 1 (installs) then Cell 2 (Drive mount) — you'll get a Google auth popup, approve it.
4. Edit **Cell 4 (Config)** — this is the only cell you should need to change between runs.
5. Run the rest top to bottom. It works today on 3 synthetic placeholder rows so you can confirm the whole pipeline before your real Week 2 dataset exists.
6. Run it **twice per model**: once with `USE_COT = False` (Week 5), once with `USE_COT = True` (Week 6) — everything else stays identical so the comparison is fair.


In [ ]:
# CELL 1 — Install dependencies (Colab-friendly versions)
!pip install -q transformers peft bitsandbytes trl datasets accelerate scikit-learn pandas
# wandb ships as a trl dependency but Colab's preinstalled version frequently has a broken
# circular import (AttributeError: partially initialized module 'wandb'...). We don't use it
# (report_to="none" below), so remove it entirely rather than fight the broken install.
# NOTE: if you've already hit the wandb AttributeError in this session, uninstalling alone
# won't fix it — you must also Runtime > Restart session, then re-run from Cell 1.
!pip uninstall -y wandb -q


In [ ]:
# CELL 2 — Mount Google Drive (Colab-safe checkpoint storage)
# If you're NOT in Colab (e.g. running locally/Jupyter), this cell auto-skips and
# checkpoints just save to a local folder instead — safe either way.
import os

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/phishing-slm-research"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print("Mounted. Project root:", DRIVE_ROOT)
else:
    DRIVE_ROOT = "."
    print("Not running in Colab — using local folder as project root:", DRIVE_ROOT)


In [ ]:
# CELL 3 — Imports and GPU check
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — go to Runtime > Change runtime type > T4 GPU, then re-run.")


In [ ]:
# CELL 4 — Config (the only cell you should need to edit between runs)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # swap for Llama-3.2-1B/3B-Instruct, Qwen2.5-3B-Instruct, Phi-3.5-mini-instruct, etc.
USE_COT = False                              # False = naive fine-tune (Week 5), True = CoT-distilled fine-tune (Week 6)

# Data path — point this at your real Week 2 dataset once it exists.
# Recommended: keep your dataset in Drive too, so it survives disconnects.
DATA_PATH = f"{DRIVE_ROOT}/data/processed/phishing_dataset.csv"

# Checkpoints save to Drive automatically — survives Colab disconnects.
OUTPUT_DIR = f"{DRIVE_ROOT}/finetuning/checkpoints/{MODEL_NAME.split('/')[-1]}-{'cot' if USE_COT else 'naive'}"

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
NUM_EPOCHS = 3

print("Data path:  ", DATA_PATH)
print("Output dir: ", OUTPUT_DIR)


In [ ]:
# CELL 5 — Load data (falls back to a tiny synthetic placeholder if your real CSV isn't ready yet)
import os

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print("Loaded real dataset:", df.shape)
else:
    print("WARNING: real dataset not found at", DATA_PATH, "— using synthetic placeholder rows for pipeline testing only.")
    df = pd.DataFrame({
        "subject": ["Urgent: Verify your account", "Team lunch Friday?", "Invoice #4471 overdue"],
        "body": [
            "Dear user, your account will be suspended unless you verify your details immediately by clicking the link below.",
            "Hey all, thinking of grabbing lunch Friday around noon, anyone in?",
            "Please find attached the overdue invoice. Payment is required within 24 hours or your service will be terminated.",
        ],
        "label": [1, 0, 1],
        "reasoning": [
            "The email creates urgency and asks the user to click a link to 'verify' account details — a classic credential-harvesting pattern. Label: phishing.",
            "This is a casual, low-stakes internal message with no request for sensitive action. Label: legitimate.",
            "The email uses a threat (service termination) and urgency (24 hours) to pressure immediate payment — a common invoice-fraud pattern. Label: phishing.",
        ],
    })

df.head()


## Prompt formatting

Naive fine-tuning trains on: `email -> label`.
CoT-distilled fine-tuning trains on: `email -> reasoning trace -> label` (the `reasoning` column should come from your Week 6 teacher-model generation step).


In [ ]:
# CELL 6 — Build instruction-formatted training text
def format_example(row):
    instruction = "Classify the following email as 'Phishing' or 'Legitimate'."
    user_msg = f"Subject: {row['subject']}\n\nBody: {row['body']}"
    label_str = "Phishing" if row["label"] == 1 else "Legitimate"

    if USE_COT and "reasoning" in row and pd.notna(row.get("reasoning", None)):
        assistant_msg = f"{row['reasoning']}\n\nFinal answer: {label_str}"
    else:
        assistant_msg = label_str

    text = f"### Instruction:\n{instruction}\n\n### Email:\n{user_msg}\n\n### Response:\n{assistant_msg}"
    return text

df["text"] = df.apply(format_example, axis=1)
train_dataset = Dataset.from_pandas(df[["text"]])
print(train_dataset[0]["text"])


In [ ]:
# CELL 7 — Load model in 4-bit (QLoRA) + tokenizer
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False


In [ ]:
# CELL 8 — Attach a LoRA adapter
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear",   # simplest safe default across model families
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# CELL 9 — Fine-tune with TRL's SFTTrainer
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=torch.cuda.is_available(),
    report_to="none",
    dataset_text_field="text",
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
)

trainer.train()


In [ ]:
# CELL 10 — Save the LoRA adapter to Drive (small — a few MB, not the full model)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved adapter to:", OUTPUT_DIR)
print("This is on Google Drive, so it survives a Colab disconnect.")


## Quick sanity-check inference

Run this after training to eyeball a couple of predictions before moving to the full evaluation harness (Week 8).


In [ ]:
# CELL 11 — Quick inference smoke test
model.eval()
test_email = "Subject: Your package could not be delivered\n\nBody: Click here within 12 hours to reschedule delivery or your package will be returned to sender."
prompt = f"### Instruction:\nClassify the following email as 'Phishing' or 'Legitimate'.\n\n### Email:\n{test_email}\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))


---
### Reloading a saved adapter later (e.g. new Colab session, evaluation script)

```python
from peft import PeftModel
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)   # OUTPUT_DIR = your Drive path
```

### Next steps
- Replace the synthetic placeholder with your real `phishing_dataset.csv` from Week 2 — put it in Drive at `phishing-slm-research/data/processed/` so this notebook finds it automatically via `DRIVE_ROOT`.
- Run this notebook twice per model — `USE_COT = False` then `USE_COT = True` — to produce the naive and CoT-distilled arms for Table 3.
- For the CoT arm, first generate the `reasoning` column with a teacher model (Week 6's distillation step) — ask for that script when you're ready.
- Keep evaluation (accuracy/F1/AUROC) in a separate `eval/evaluate.py`, not this training notebook.
